# Bagian A final — satu model, dievaluasi pada data uji bebas duplikat (70:15:15)

Notebook ini menggantikan Bagian A di `fundus_effb0_se_sesuai_artikel.ipynb` untuk naskah revisi. Model, augmentasi, jadwal LR, dan metrik **sama**; yang berbeda:

| # | Bagian A lama | Notebook ini | Alasan |
|---|---|---|---|
| 1 | Pembagian 90:10, evaluasi pada data validasi | Pembagian **70:15:15** yang identik dengan Bagian B (grup duplikat pHash tidak dipisah), evaluasi pada **data uji** | Data validasi dipakai untuk memilih epoch, jadi tidak boleh dipakai untuk angka akhir (Reviewer A#5); 95 citra validasi lama punya kembaran di data latih |
| 2 | Early stopping dan bobot terbaik memantau `val_loss` | Memantau **`val_accuracy`** (sama dengan Bagian B, 15 Nov) | `val_loss` naik turun tajam; akurasi uji berkorelasi 0,78 dengan epoch terbaik pada 30 run pertama |
| 3 | Kriteria henti tambahan `val_acc ≥ 0,95 & val_loss < 0,13` | **Dimatikan** | Dikritik reviewer; tidak pernah terpenuhi |
| 4 | Waktu inferensi diukur dengan pemanggilan *eager* | Diukur dengan fungsi terkompilasi (`tf.function`), batch 1 dan 32 | Angka lama (178 ms GPU) adalah artefak pengukuran |

Perbaikan lama tetap berlaku: augmentasi benar-benar dipakai, Keras 2 lewat `tf-keras`, BatchNorm *backbone* dalam mode inferensi, seed global.

**Cara pakai di Kaggle**: Accelerator GPU (T4), Internet ON, dataset *eye-diseases-classification* terpasang, lalu **Save Version → Save & Run All (Commit)**. Lama sekitar 1 jam. Semua keluaran ada di tab Output, folder `hasil_A`.


## Konfigurasi


### A0. Paksa Keras 2 (wajib di Kaggle/Colab)

Kaggle dan Colab kini memakai **Keras 3**. Uji di laptop (Keras 2.15 vs Keras 3, bobot ImageNet identik) menunjukkan keluaran EfficientNetB0 pretrained di Keras 3 **berbeda jauh** (korelasi fitur 0,84; kelas ImageNet berbeda pada 6 dari 8 citra), sehingga fine-tune hanya mencapai ±63% validasi. Sel ini memasang `tf-keras` (Keras 2 klasik) dan mengaktifkannya lewat `TF_USE_LEGACY_KERAS=1`. Di TensorFlow 2.15 (lokal) sel ini tidak berbuat apa-apa. **Jalankan sel ini pertama, sebelum `import tensorflow`.**

In [ ]:
import os, sys, subprocess, importlib.util
from importlib.metadata import version, PackageNotFoundError
try:
    _tf = tuple(int(v) for v in version('tensorflow').split('.')[:2])
except PackageNotFoundError:
    _tf = (0, 0)
if _tf >= (2, 16):                                # TF >= 2.16 bawaannya Keras 3 -> butuh tf-keras
    os.environ['TF_USE_LEGACY_KERAS'] = '1'      # harus diset SEBELUM import tensorflow; jangan diset di TF 2.15
    if importlib.util.find_spec('tf_keras') is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', f'tf-keras~={_tf[0]}.{_tf[1]}.0'], check=True)
        print('tf-keras dipasang untuk TensorFlow', '.'.join(map(str, _tf)))
    import tf_keras
    print('Keras 2 aktif (tf_keras', tf_keras.__version__ + ')')
else:
    print('TensorFlow', '.'.join(map(str, _tf)), '-> sudah Keras 2, tidak perlu tf-keras')

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
_CANDS = ['/kaggle/input/eye-diseases-classification/dataset',
          '/kaggle/input/datasets/gunavenkatdoddi/eye-diseases-classification/dataset']
DATASET_DIR = next((d for d in _CANDS if os.path.isdir(d)), None)
if DATASET_DIR is None:
    import kagglehub
    DATASET_DIR = os.path.join(kagglehub.dataset_download('gunavenkatdoddi/eye-diseases-classification'), 'dataset')
OUT_DIR = '/kaggle/working/hasil_A' if os.path.isdir('/kaggle/working') else 'hasil_A'

SEED        = 123          # seed pembagian data DAN seed model (ditetapkan sebelum melihat hasil; sama dengan naskah)
IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 150
VAL_SPLIT   = 0.10         # hanya untuk mereproduksi urutan berkas dan tabel 90:10 naskah awal; pelatihan memakai 70:15:15
ATENSI      = 'se'         # 'se' = model di judul artikel. 'none' = EfficientNetB0 tanpa blok tambahan
SE_RATIO    = 8
PANTAU      = 'val_accuracy'   # besaran untuk early stopping dan bobot terbaik (sama dengan Bagian B)
PATIENCE    = 30
USE_STOP_CRITERION    = False  # kriteria val_acc >= 0,95 & val_loss < 0,13 dimatikan
BN_BACKBONE_INFERENCE = True
LR_PER_ITERASI        = False
GLAUCOMA_PREVALENCE   = 0.0354   # Tham dkk. [3], usia 40 sampai 80 tahun
UJI_CEPAT   = False        # True: uji alur beberapa menit (sedikit citra, 2 epoch)
if UJI_CEPAT: EPOCHS = 2
os.makedirs(OUT_DIR, exist_ok=True)
print('DATASET_DIR :', DATASET_DIR)
print('OUT_DIR     :', os.path.abspath(OUT_DIR))


### A1. Pustaka, info lingkungan, dan seed global (untuk placeholder artikel)

In [ ]:
import json, time, random, platform, glob, math, itertools, importlib.metadata
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import tensorflow as tf
keras = tf.keras
import sys
if 'tf_keras' in sys.modules:      # Keras 2 lewat paket tf-keras (sel A0)
    keras_version = 'tf-keras ' + importlib.metadata.version('tf-keras')
else:
    keras_version = getattr(keras, '__version__', None) or importlib.metadata.version('keras')
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score, classification_report
from sklearn.preprocessing import label_binarize

tf.keras.utils.set_random_seed(SEED)       # Python, NumPy, dan TensorFlow
gpus = tf.config.list_physical_devices('GPU')
gpu_name = tf.config.experimental.get_device_details(gpus[0]).get('device_name', 'GPU') if gpus else 'CPU saja'
ENV = {'python': platform.python_version(), 'tensorflow': tf.__version__, 'keras': keras_version,
       'gpu': gpu_name, 'numpy': np.__version__}
json.dump(ENV, open(f'{OUT_DIR}/lingkungan.json', 'w'), indent=1)
print(ENV)

### A2. Daftar berkas (urutan identik dengan notebook lama, seed 123)
Pembagian 90:10 di sini hanya dipakai untuk melaporkan kebocoran duplikat pada pembagian naskah awal. Pelatihan memakai pembagian 70:15:15 di sel A3b.


In [ ]:
common = dict(validation_split=VAL_SPLIT, seed=SEED, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
              shuffle=True, label_mode='int')
train_raw = keras.utils.image_dataset_from_directory(DATASET_DIR, subset='training', **common)
val_raw   = keras.utils.image_dataset_from_directory(DATASET_DIR, subset='validation', **common)
class_names = train_raw.class_names
NUM_CLASSES = len(class_names)
CLS_ID = {c: i for i, c in enumerate(class_names)}
GLAUCOMA = CLS_ID['glaucoma']; NORMAL = CLS_ID['normal']

train_files = list(train_raw.file_paths); val_files = list(val_raw.file_paths)
assert not set(train_files) & set(val_files), 'ada berkas yang muncul di latih dan validasi!'
lbl = lambda p: os.path.basename(os.path.dirname(p))
split_df = pd.DataFrame({'file': train_files + val_files,
                         'kelas': [lbl(p) for p in train_files + val_files],
                         'subset': ['latih'] * len(train_files) + ['validasi'] * len(val_files)})
split_df.to_csv(f'{OUT_DIR}/pembagian_90_10_naskah_awal.csv', index=False)
tabel2 = pd.crosstab(split_df.kelas, split_df.subset, margins=True, margins_name='Total')
print(tabel2)
print('\nNaskah awal (Tabel 2): latih 940/985/900/971 = 3.796; validasi 98/113/107/103 = 421')

### A3. Karakteristik citra per kelas dan deteksi duplikat (Reviewer A#4)
Ukuran asli citra dan pola nama berkas per kelas memberi petunjuk sumber dataset; *perceptual hash* mendeteksi citra (hampir) identik, termasuk yang muncul di latih **dan** validasi.

In [ ]:
from scipy.fftpack import dct
def phash(path, hash_size=8, highfreq=4):
    img = Image.open(path).convert('L').resize((hash_size * highfreq,) * 2, Image.LANCZOS)
    a = np.asarray(img, dtype=np.float32)
    d = dct(dct(a, axis=0, norm='ortho'), axis=1, norm='ortho')[:hash_size, :hash_size]
    bits = (d > np.median(d)).flatten()
    return np.packbits(bits).view('>u8')[0]

rows = []
for p in split_df.file:
    with Image.open(p) as im:
        w, h = im.size
    rows.append((w, h, os.path.splitext(p)[1].lower(), phash(p)))
_meta = pd.DataFrame(rows, index=split_df.index, columns=['lebar', 'tinggi', 'ekstensi', 'phash'])
for _c in _meta.columns: split_df[_c] = _meta[_c]
split_df['pola_nama'] = split_df.file.map(lambda p: 'ODIR-like (_left/_right)' if ('_left' in p or '_right' in p)
                                          else ('angka_angka' if os.path.basename(p).startswith('_') else 'lainnya'))
ukuran = split_df.groupby('kelas').agg(n=('file', 'size'), lebar_median=('lebar', 'median'), tinggi_median=('tinggi', 'median'),
                                       lebar_min=('lebar', 'min'), lebar_maks=('lebar', 'max'))
pola = pd.crosstab(split_df.kelas, split_df.pola_nama)
ekst = pd.crosstab(split_df.kelas, split_df.ekstensi)
print(ukuran, '\n\n', pola, '\n\n', ekst)
ukuran.join(pola).join(ekst).to_csv(f'{OUT_DIR}/karakteristik_citra_per_kelas.csv')

H = split_df.phash.values.astype(np.uint64)
def hamming(a, B):
    x = np.bitwise_xor(B, a)
    return np.unpackbits(x.view(np.uint8).reshape(-1, 8), axis=1).sum(1)
THR = 4   # jarak Hamming <= 4 dari 64 bit dianggap duplikat/hampir duplikat
pairs = []
for i in range(len(H)):
    d = hamming(H[i], H[i + 1:])
    for j in np.nonzero(d <= THR)[0]:
        pairs.append((i, i + 1 + j, int(d[j])))
dup = pd.DataFrame(pairs, columns=['i', 'j', 'hamming'])
if len(dup):
    dup['kelas_i'] = split_df.kelas.values[dup.i]; dup['kelas_j'] = split_df.kelas.values[dup.j]
    dup['subset_i'] = split_df.subset.values[dup.i]; dup['subset_j'] = split_df.subset.values[dup.j]
    dup['file_i'] = split_df.file.values[dup.i]; dup['file_j'] = split_df.file.values[dup.j]
dup.to_csv(f'{OUT_DIR}/pasangan_duplikat.csv', index=False)
lintas = dup[(dup.subset_i != dup.subset_j)] if len(dup) else dup
beda_kelas = dup[(dup.kelas_i != dup.kelas_j)] if len(dup) else dup
print(f'Pasangan (hampir) duplikat: {len(dup)} | lintas latih-validasi: {len(lintas)} | label berbeda: {len(beda_kelas)}')
print('Citra validasi yang punya kembaran di data latih:', len(set(lintas.i).union(lintas.j) & set(np.nonzero(split_df.subset.values == "validasi")[0])) if len(lintas) else 0)

### A3b. Pembagian 70:15:15 bebas duplikat (identik dengan Bagian B)
Citra yang (hampir) kembar selalu ditempatkan di subset yang sama. Grup yang anggotanya berlabel berbeda dibuang.


In [ ]:
from sklearn.model_selection import train_test_split
# kelompok duplikat (union-find) -> pembagian 70:15:15 terstratifikasi per grup. IDENTIK dengan sel B1 notebook Bagian B.
parent = list(range(len(split_df)))
def find(a):
    while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
    return a
for i, j in zip(dup.i if len(dup) else [], dup.j if len(dup) else []): parent[find(i)] = find(j)
split_df['grup'] = [find(i) for i in range(len(split_df))]
konflik = split_df.groupby('grup').kelas.nunique(); konflik = konflik[konflik > 1].index
print('Grup duplikat dengan label berbeda (dibuang):', len(konflik), '| citra dibuang:', int(split_df.grup.isin(konflik).sum()))
dfB = split_df[~split_df.grup.isin(konflik)].copy()
grp = dfB.groupby('grup').agg(kelas=('kelas', 'first')).reset_index()
g_tr, g_tmp = train_test_split(grp, test_size=0.30, stratify=grp.kelas, random_state=SEED)
g_va, g_te = train_test_split(g_tmp, test_size=0.50, stratify=g_tmp.kelas, random_state=SEED)
dfB['subset'] = dfB.grup.map({**{g: 'latih' for g in g_tr.grup}, **{g: 'validasi' for g in g_va.grup}, **{g: 'uji' for g in g_te.grup}})
dfB.to_csv(f'{OUT_DIR}/pembagian_70_15_15.csv', index=False)
tabel2_baru = pd.crosstab(dfB.kelas, dfB.subset, margins=True, margins_name='Total')[['latih', 'validasi', 'uji', 'Total']]
print(tabel2_baru); tabel2_baru.to_csv(f'{OUT_DIR}/tabel2_pembagian_70_15_15.csv')
train_files, val_files, test_files = (dfB.file[dfB.subset == s].tolist() for s in ['latih', 'validasi', 'uji'])
assert not (set(train_files) & set(test_files)) and not (set(val_files) & set(test_files)) and not (set(train_files) & set(val_files))
assert not (set(dfB.grup[dfB.subset == 'latih']) & set(dfB.grup[dfB.subset == 'uji'])), 'grup duplikat terpisah antarsubset!'
if not UJI_CEPAT:
    assert (len(train_files), len(val_files), len(test_files)) == (2648, 560, 555), 'pembagian berbeda dari Bagian B: periksa versi dataset'
else:
    train_files, val_files, test_files = (v[::max(1, len(v) // n)][:n] for v, n in [(train_files, 96), (val_files, 48), (test_files, 48)])
eval_files = test_files          # SEMUA angka akhir dihitung pada data uji; data validasi hanya untuk memilih epoch
print('latih / validasi / uji:', len(train_files), len(val_files), len(test_files))


### A4. Pipeline data
Evaluasi memakai *dataset* berurutan dari daftar berkas uji sehingga label, prediksi, dan nama berkas selalu sejajar.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
def load_img(path):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, method='bilinear')     # sama dengan image_dataset_from_directory
    img.set_shape(IMG_SIZE + (3,))
    return img

def ds_from_paths(paths, training=False, seed=SEED):
    labels = [CLS_ID[lbl(p)] for p in paths]
    ds = tf.data.Dataset.from_tensor_slices((list(paths), labels))
    ds = ds.map(lambda p, y: (load_img(p), tf.one_hot(y, NUM_CLASSES)), num_parallel_calls=AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(1000, seed=seed, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = ds_from_paths(train_files, training=True, seed=SEED)    # sama dengan Bagian B
val_ds   = ds_from_paths(val_files)                                # hanya untuk early stopping / memilih epoch
del train_raw, val_raw

### A5. Augmentasi (Tabel 3 artikel)
Catatan reviewer A#11: derau Gaussian σ = 0,05 pada skala piksel 0 sampai 255 praktis tidak berpengaruh. Nilai dipertahankan agar sama dengan artikel; ubah `NOISE_STD` (mis. 0,05 × 255 ≈ 12,75) bila ingin derau yang bermakna, lalu laporkan di artikel.

In [ ]:
NOISE_STD = 0.05

@keras.utils.register_keras_serializable(package='fundus')   # keras.utils ada di Keras 2 dan 3 (keras.saving tidak dimuat lewat tf.keras di Keras 3)
class RandomCropResize(keras.layers.Layer):
    """Potong acak crop×crop lalu ubah kembali ke ukuran semula; HANYA saat pelatihan."""
    def __init__(self, crop=200, out=224, seed=None, **kw):
        super().__init__(**kw); self.crop, self.out, self.seed = crop, out, seed
    def call(self, x, training=None):
        if not training:
            return x
        b = tf.shape(x)[0]
        x = tf.image.random_crop(x, tf.stack([b, self.crop, self.crop, 3]), seed=self.seed)
        return tf.image.resize(x, (self.out, self.out))
    def get_config(self):
        return {**super().get_config(), 'crop': self.crop, 'out': self.out, 'seed': self.seed}

def make_augmentation(seed=SEED):
    return keras.Sequential([
        RandomCropResize(200, 224, seed=seed),
        keras.layers.RandomFlip('horizontal', seed=seed),
        keras.layers.RandomRotation(0.1, seed=seed),          # faktor 0,1 = ±36°
        keras.layers.RandomZoom(0.1, seed=seed),
        keras.layers.RandomContrast(0.1, seed=seed),
        keras.layers.RandomBrightness(0.1, seed=seed),
        keras.layers.RandomTranslation(0.1, 0.1, seed=seed),
        keras.layers.GaussianNoise(NOISE_STD, seed=seed),
    ], name='augmentasi')

### A6. Arsitektur (Gambar 1 artikel)
Semua lapisan kepala diberi nama agar dapat dipakai ulang untuk Grad-CAM. Fungsi yang sama mendukung varian ablasi (Bagian B).

In [ ]:
L = keras.layers

def attention_block(x, kind='se', ratio=8):
    C = x.shape[-1]
    if kind == 'none':
        return x
    if kind == 'se':
        s = L.GlobalAveragePooling2D(name='se_squeeze')(x)
        s = L.Reshape((1, 1, C), name='se_reshape')(s)
        s = L.Dense(C // ratio, activation='relu', name='se_fc1')(s)
        s = L.Dense(C, activation='sigmoid', name='se_fc2')(s)
        return L.Multiply(name='se_scale')([x, s])
    if kind == 'eca':           # Wang dkk. (ECA-Net), k adaptif
        k = int(abs((math.log2(C) + 1) / 2)); k = k if k % 2 else k + 1
        s = L.GlobalAveragePooling2D(name='eca_gap')(x)
        s = L.Reshape((C, 1), name='eca_reshape1')(s)
        s = L.Conv1D(1, k, padding='same', use_bias=False, name='eca_conv')(s)
        s = L.Activation('sigmoid', name='eca_sigmoid')(s)
        s = L.Reshape((1, 1, C), name='eca_reshape2')(s)
        return L.Multiply(name='eca_scale')([x, s])
    if kind == 'cbam':          # Woo dkk. (CBAM): atensi kanal lalu spasial
        fc1 = L.Dense(C // ratio, activation='relu', name='cbam_fc1'); fc2 = L.Dense(C, name='cbam_fc2')
        a = fc2(fc1(L.GlobalAveragePooling2D(name='cbam_gap')(x)))
        m = fc2(fc1(L.GlobalMaxPooling2D(name='cbam_gmp')(x)))
        ca = L.Activation('sigmoid', name='cbam_ch_sig')(L.Add(name='cbam_ch_add')([a, m]))
        x = L.Multiply(name='cbam_ch_scale')([x, L.Reshape((1, 1, C), name='cbam_ch_reshape')(ca)])
        avg = L.Lambda(lambda t: tf.reduce_mean(t, -1, keepdims=True), name='cbam_sp_mean')(x)
        mx = L.Lambda(lambda t: tf.reduce_max(t, -1, keepdims=True), name='cbam_sp_max')(x)
        sa = L.Conv2D(1, 7, padding='same', activation='sigmoid', name='cbam_sp_conv')(L.Concatenate(name='cbam_sp_cat')([avg, mx]))
        return L.Multiply(name='cbam_sp_scale')([x, sa])
    raise ValueError(kind)

def make_backbone(name):
    if name == 'efficientnetb0':
        return keras.applications.EfficientNetB0(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)), None
    if name == 'resnet50':
        return keras.applications.ResNet50(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)), keras.applications.resnet50.preprocess_input
    if name == 'mobilenetv2':
        return keras.applications.MobileNetV2(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,)), keras.applications.mobilenet_v2.preprocess_input
    raise ValueError(name)

def build_model(backbone='efficientnetb0', attention='se', ratio=SE_RATIO, seed=SEED):
    base, prep = make_backbone(backbone)
    base.trainable = True                      # seluruh lapisan di-fine-tune
    inputs = L.Input(IMG_SIZE + (3,), name='citra')
    x = make_augmentation(seed)(inputs)        # PERBAIKAN #1: keluaran augmentasi dipakai
    if prep is not None:                        # EfficientNet Keras sudah memuat rescaling+normalisasi
        x = L.Lambda(prep, name='preprocess')(x)
    # PERBAIKAN #2: training tidak dikunci ke True. PERBAIKAN #9: BN backbone dalam mode inferensi (bobot tetap dilatih)
    x = base(x, training=False) if BN_BACKBONE_INFERENCE else base(x)
    x = attention_block(x, attention, ratio)
    x = L.GlobalAveragePooling2D(name='head_gap')(x)
    x = L.BatchNormalization(name='head_bn1')(x)
    x = L.Dense(128, activation='relu', name='head_fc')(x)
    x = L.BatchNormalization(name='head_bn2')(x)
    x = L.Dropout(0.3, name='head_dropout')(x)
    out = L.Dense(NUM_CLASSES, activation='softmax', name='keluaran')(x)
    return keras.Model(inputs, out, name=f'{backbone}_{attention}{ratio if attention in ("se","cbam") else ""}'), base

model, base_model = build_model(attention=ATENSI)
n_total = model.count_params()
n_train = int(sum(np.prod(v.shape) for v in model.trainable_weights))
n_se = sum(model.get_layer(n).count_params() for n in ['se_fc1', 'se_fc2']) if ATENSI == 'se' else 0
print(f'Total parameter: {n_total:,} | dapat dilatih: {n_train:,} | blok SE: {n_se:,} ({100*n_se/n_total:.1f}%)'.replace(',', '.'))
print('Naskah (SE r=8): total 4.630.727; dapat dilatih 4.585.888; blok SE 411.040')

### A7. Bobot kelas, jadwal laju pembelajaran (Persamaan (4) dan (5)), dan *callback*

In [ ]:
y_tr = np.array([CLS_ID[lbl(p)] for p in train_files])
counts = np.bincount(y_tr, minlength=NUM_CLASSES)
class_weight = {i: len(y_tr) / (NUM_CLASSES * c) for i, c in enumerate(counts)}
print('Bobot kelas:', {class_names[i]: round(w, 3) for i, w in class_weight.items()}, '| naskah awal (90:10): 1,010 / 0,963 / 1,054 / 0,977')

def cyclic_lr(epoch, lr=None, base_lr=1e-5, max_lr=1e-3, step_size=2000):
    cycle = np.floor(1 + epoch / (2 * step_size))
    x = np.abs(epoch / step_size - 2 * cycle + 1)
    return float(base_lr + (max_lr - base_lr) * max(0, 1 - x))
class CyclicLRIter(keras.callbacks.Callback):
    """Persamaan (4) dan (5) dengan t = nomor iterasi (batch), bukan nomor epoch."""
    def on_train_begin(self, logs=None): self.it = 0
    def on_train_batch_begin(self, batch, logs=None):
        keras.backend.set_value(self.model.optimizer.learning_rate, cyclic_lr(self.it)); self.it += 1
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None: logs['lr'] = float(keras.backend.get_value(self.model.optimizer.learning_rate))

_spe = math.ceil(len(train_files) / BATCH_SIZE)
if LR_PER_ITERASI:
    print(f'LR per iterasi ({_spe} iterasi/epoch, setengah siklus = 2000 iterasi ≈ {2000/_spe:.1f} epoch): '
          + '; '.join(f'akhir epoch {e} = {cyclic_lr(e*_spe-1):.2e}' for e in (1, 5, 10, 17, 25, 34)))
else:
    print(f'LR epoch 1 = {cyclic_lr(0):.3e}; epoch 150 = {cyclic_lr(149):.3e} (η_max tidak tercapai -> pemanasan linear)')

class StopCriterion(keras.callbacks.Callback):
    def __init__(self): super().__init__(); self.fired_epoch = None
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        if logs.get('val_accuracy', 0) >= 0.95 and logs.get('val_loss', 9) < 0.13:
            self.fired_epoch = epoch + 1; self.model.stop_training = True

class Timer(keras.callbacks.Callback):
    def on_train_begin(self, logs=None): self.t0 = time.time()
    def on_train_end(self, logs=None): self.seconds = time.time() - self.t0

class BestKeeper(keras.callbacks.Callback):
    """Simpan bobot epoch terbaik di memori dan pasang kembali di akhir pelatihan, juga bila early stopping TIDAK terpicu."""
    def __init__(self, monitor):
        super().__init__(); self.monitor = monitor; self.sign = -1 if 'loss' in monitor else 1
        self.best = -np.inf; self.best_epoch = 0; self.w = None
    def on_epoch_end(self, epoch, logs=None):
        v = self.sign * float((logs or {}).get(self.monitor, -np.inf * self.sign))
        if v > self.best: self.best, self.best_epoch, self.w = v, epoch + 1, self.model.get_weights()
    def on_train_end(self, logs=None):
        if self.w is not None: self.model.set_weights(self.w)

def make_callbacks(ckpt_path=None, use_stop=USE_STOP_CRITERION, log_path=None):
    stop, timer = StopCriterion(), Timer()
    es = keras.callbacks.EarlyStopping(monitor=PANTAU, mode=('min' if 'loss' in PANTAU else 'max'), patience=PATIENCE)
    es.keeper = BestKeeper(PANTAU)
    cbs = [CyclicLRIter() if LR_PER_ITERASI else keras.callbacks.LearningRateScheduler(cyclic_lr), es, es.keeper, timer]
    if use_stop: cbs.insert(0, stop)
    if log_path: cbs.append(keras.callbacks.CSVLogger(log_path))
    return cbs, stop, es, timer

### A8. Pelatihan

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
              loss='categorical_crossentropy', metrics=['accuracy'])
cbs, stop_cb, es_cb, timer_cb = make_callbacks(log_path=f'{OUT_DIR}/riwayat_pelatihan.csv')
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, class_weight=class_weight, callbacks=cbs, verbose=2)

hist = pd.DataFrame(history.history); hist.index += 1
best_epoch = int(es_cb.keeper.best_epoch)          # bobot epoch ini sudah dipasang kembali ke model
TRAIN_INFO = {'epoch_ditempuh': int(len(hist)), 'epoch_terbaik': best_epoch, 'dipantau': PANTAU,
              'val_akurasi_epoch_terbaik': float(hist.val_accuracy[best_epoch]), 'val_loss_epoch_terbaik': float(hist.val_loss[best_epoch]),
              'berhenti_karena': (f'early stopping ({PANTAU}, patience {PATIENCE})' if len(hist) < EPOCHS else 'epoch maksimum'),
              'waktu_latih_menit': round(timer_cb.seconds / 60, 1),
              'lr_epoch_terakhir': float(hist['learning_rate'].iloc[-1] if 'learning_rate' in hist else hist['lr'].iloc[-1]) if ('learning_rate' in hist or 'lr' in hist) else None}
json.dump(TRAIN_INFO, open(f'{OUT_DIR}/info_pelatihan.json', 'w'), indent=1)
print(TRAIN_INFO)
best_path = f'{OUT_DIR}/model_terbaik.weights.h5'
model.save_weights(best_path)                      # bobot terbaik; arsitektur dibangun ulang dengan build_model()


### A9. Kurva pelatihan (gambar tambahan untuk artikel, Reviewer A#10)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for k, a in zip(['accuracy', 'loss'], ax):
    a.plot(hist.index, hist[k], label='Latih'); a.plot(hist.index, hist['val_' + k], label='Validasi')
    a.axvline(best_epoch, ls='--', c='gray', label=f'Epoch terbaik ({best_epoch})')
    a.set_xlabel('Epoch'); a.set_ylabel('Akurasi' if k == 'accuracy' else 'Loss'); a.legend(); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/kurva_pelatihan.png', dpi=300); plt.show()

### A10. Prediksi data uji (bobot terbaik) dan simpan probabilitas per citra


In [ ]:
def predict_paths(m, paths):
    ds = ds_from_paths(paths)
    probs = m.predict(ds, verbose=0)
    y = np.array([CLS_ID[lbl(p)] for p in paths])
    return y, probs

y_val, probs_val = predict_paths(model, val_files)
print('Akurasi validasi (dipakai untuk memilih epoch):', (y_val == probs_val.argmax(1)).mean().round(4), '| n =', len(y_val))
y_true, probs = predict_paths(model, eval_files)
y_pred = probs.argmax(1)
pred_df = pd.DataFrame(probs, columns=[f'p_{c}' for c in class_names])
pred_df.insert(0, 'file', eval_files); pred_df.insert(1, 'label', [class_names[i] for i in y_true])
pred_df.insert(2, 'prediksi', [class_names[i] for i in y_pred])
pred_df.to_csv(f'{OUT_DIR}/prediksi_uji.csv', index=False)
np.save(f'{OUT_DIR}/prob_validasi.npy', probs_val)
print('Akurasi UJI:', (y_true == y_pred).mean().round(4), '| n =', len(y_true))

### A11. Fungsi metrik (CI Wilson, *bootstrap*, AUC, kalibrasi, PPV)

In [ ]:
Z = 1.959963984540054
def wilson(k, n):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + Z**2 / n; c = (p + Z**2 / (2 * n)) / d
    h = Z * math.sqrt(p * (1 - p) / n + Z**2 / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def confusion(y, yp, k):
    cm = np.zeros((k, k), int); np.add.at(cm, (y, yp), 1); return cm

def per_class(cm):
    n = cm.sum(); out = []
    for i in range(len(cm)):
        tp = cm[i, i]; fn = cm[i].sum() - tp; fp = cm[:, i].sum() - tp; tn = n - tp - fn - fp
        prec = tp / (tp + fp) if tp + fp else 0.0; rec = tp / (tp + fn) if tp + fn else 0.0
        out.append(dict(tp=tp, fp=fp, fn=fn, tn=tn, presisi=prec, sensitivitas=rec,
                        spesifisitas=tn / (tn + fp) if tn + fp else 0.0,
                        f1=2 * prec * rec / (prec + rec) if prec + rec else 0.0, support=tp + fn))
    return out

def kappa(cm):
    n = cm.sum(); po = np.trace(cm) / n; pe = (cm.sum(0) * cm.sum(1)).sum() / n**2
    return (po - pe) / (1 - pe)

def summary_metrics(y, yp, k):
    cm = confusion(y, yp, k); pc = per_class(cm); sup = np.array([d['support'] for d in pc])
    macro = {m: np.mean([d[m] for d in pc]) for m in ['presisi', 'sensitivitas', 'spesifisitas', 'f1']}
    weighted = {m: np.average([d[m] for d in pc], weights=sup) for m in ['presisi', 'sensitivitas', 'spesifisitas', 'f1']}
    return dict(cm=cm, per_class=pc, macro=macro, weighted=weighted, akurasi=np.trace(cm) / cm.sum(), kappa=kappa(cm))

def bootstrap(y, probs, B=2000, seed=SEED):
    rng = np.random.default_rng(seed); n = len(y); k = probs.shape[1]; yb = label_binarize(y, classes=range(k))
    res = {'kappa': [], 'macro_f1': [], 'akurasi': [], 'auc_micro': [], **{f'auc_{i}': [] for i in range(k)}}
    for _ in range(B):
        idx = rng.integers(0, n, n); s = summary_metrics(y[idx], probs[idx].argmax(1), k)
        res['kappa'].append(s['kappa']); res['macro_f1'].append(s['macro']['f1']); res['akurasi'].append(s['akurasi'])
        if len(np.unique(y[idx])) == k:
            res['auc_micro'].append(roc_auc_score(yb[idx], probs[idx], average='micro'))
            for i in range(k): res[f'auc_{i}'].append(roc_auc_score(yb[idx, i], probs[idx, i]))
    return {m: tuple(np.percentile(v, [2.5, 97.5])) for m, v in res.items() if len(v)}

def ece_score(y, probs, bins=15):
    conf = probs.max(1); correct = probs.argmax(1) == y; edges = np.linspace(0, 1, bins + 1); e = 0.0; rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean()); rows.append((conf[m].mean(), correct[m].mean(), m.sum()))
    return e, rows

def ppv_npv(sens, spec, prev):
    ppv = sens * prev / (sens * prev + (1 - spec) * (1 - prev))
    npv = spec * (1 - prev) / (spec * (1 - prev) + (1 - sens) * prev)
    return ppv, npv

fmt = lambda x, d=2: f'{x:.{d}f}'.replace('.', ',')
pct = lambda x, d=2: fmt(100 * x, d)

### A12. Tabel kinerja per kelas, *confusion matrix*, dan selang kepercayaan (data uji)


In [ ]:
S = summary_metrics(y_true, y_pred, NUM_CLASSES)
BS = bootstrap(y_true, probs)
yb = label_binarize(y_true, classes=range(NUM_CLASSES))
AUC = {i: roc_auc_score(yb[:, i], probs[:, i]) for i in range(NUM_CLASSES)}
AUC['micro'] = roc_auc_score(yb, probs, average='micro')

nama = {'cataract': 'Katarak', 'diabetic_retinopathy': 'Retinopati diabetik', 'glaucoma': 'Glaukoma', 'normal': 'Normal'}
t5 = pd.DataFrame([{'Kelas': nama[c], 'Presisi (%)': pct(d['presisi']), 'Sensitivitas (%)': pct(d['sensitivitas']),
                    'Spesifisitas (%)': pct(d['spesifisitas']), 'F1-score (%)': pct(d['f1']), 'Support': d['support']}
                   for c, d in zip(class_names, S['per_class'])])
for nm, agg in [('Macro average', S['macro']), ('Weighted average', S['weighted'])]:
    t5.loc[len(t5)] = [nm, pct(agg['presisi']), pct(agg['sensitivitas']), pct(agg['spesifisitas']), pct(agg['f1']), len(y_true)]
t5.loc[len(t5)] = ['Akurasi', '', '', '', pct(S['akurasi']), len(y_true)]
t6 = pd.DataFrame(S['cm'], index=[nama[c] for c in class_names], columns=[nama[c] for c in class_names])
t6.index.name = 'Label sebenarnya \\ Prediksi'

acc_ci = wilson(np.trace(S['cm']), S['cm'].sum())
t7 = [('Akurasi (%)', pct(S['akurasi']), f"{pct(acc_ci[0])} sampai {pct(acc_ci[1])}", 'Wilson'),
      ('Macro F1-score (%)', pct(S['macro']['f1']), f"{pct(BS['macro_f1'][0])} sampai {pct(BS['macro_f1'][1])}", 'Bootstrap'),
      ("Cohen's kappa", fmt(S['kappa'], 3), f"{fmt(BS['kappa'][0], 3)} sampai {fmt(BS['kappa'][1], 3)}", 'Bootstrap')]
for c, d in zip(class_names, S['per_class']):
    lo, hi = wilson(d['tp'], d['support']); t7.append((f'Sensitivitas {nama[c].lower()} (%)', pct(d['sensitivitas']), f'{pct(lo)} sampai {pct(hi)}', 'Wilson'))
g = S['per_class'][GLAUCOMA]; lo, hi = wilson(g['tn'], g['tn'] + g['fp'])
t7.append(('Spesifisitas glaukoma (%)', pct(g['spesifisitas']), f'{pct(lo)} sampai {pct(hi)}', 'Wilson'))
for i, c in enumerate(class_names):
    t7.append((f'AUC {nama[c].lower()}', fmt(AUC[i], 3), f"{fmt(BS[f'auc_{i}'][0], 3)} sampai {fmt(BS[f'auc_{i}'][1], 3)}", 'Bootstrap'))
t7.append(('AUC micro-average', fmt(AUC['micro'], 3), f"{fmt(BS['auc_micro'][0], 3)} sampai {fmt(BS['auc_micro'][1], 3)}", 'Bootstrap'))
t7 = pd.DataFrame(t7, columns=['Metrik', 'Estimasi', 'Selang kepercayaan 95%', 'Metode'])

with pd.ExcelWriter(f'{OUT_DIR}/tabel_artikel.xlsx') as xw:
    t5.to_excel(xw, sheet_name='Tabel 5 kinerja', index=False); t6.to_excel(xw, sheet_name='Tabel 6 confusion'); t7.to_excel(xw, sheet_name='Tabel 7 CI', index=False)
display(t5); display(t6); display(t7)

### A13. *Confusion matrix* dan kurva ROC (data uji; AUC 3 desimal)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5.2))
cmn = S['cm'] / S['cm'].sum(1, keepdims=True)
annot = np.array([[f"{S['cm'][i, j]}\n({pct(cmn[i, j], 1)}%)" for j in range(NUM_CLASSES)] for i in range(NUM_CLASSES)])
sns.heatmap(cmn, annot=annot, fmt='', cmap='Blues', cbar_kws={'label': 'Proporsi per kelas sebenarnya'},
            xticklabels=[nama[c] for c in class_names], yticklabels=[nama[c] for c in class_names], ax=ax[0])
ax[0].set_xlabel('Label prediksi'); ax[0].set_ylabel('Label sebenarnya'); ax[0].set_title('(a)', y=-0.28)
for i, c in enumerate(class_names):
    f, t, _ = roc_curve(yb[:, i], probs[:, i]); ax[1].plot(f, t, label=f'{nama[c]} (AUC = {AUC[i]:.3f})')
f, t, _ = roc_curve(yb.ravel(), probs.ravel()); ax[1].plot(f, t, 'k--', label=f"Micro-average (AUC = {AUC['micro']:.3f})")
ax[1].plot([0, 1], [0, 1], c='gray', lw=.8); ax[1].set_xlabel('False Positive Rate'); ax[1].set_ylabel('True Positive Rate')
ax[1].legend(loc='lower right', fontsize=8); ax[1].set_title('(b)', y=-0.28); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/gambar2_cm_roc.png', dpi=300, bbox_inches='tight'); plt.show()

### A14. Kurva *precision-recall* dan kalibrasi (Reviewer A#13, A#15)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
AP = {}
for i, c in enumerate(class_names):
    p, r, _ = precision_recall_curve(yb[:, i], probs[:, i]); AP[c] = average_precision_score(yb[:, i], probs[:, i])
    ax[0].plot(r, p, label=f'{nama[c]} (AP = {AP[c]:.3f})')
ax[0].set_xlabel('Recall (sensitivitas)'); ax[0].set_ylabel('Presisi'); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
ECE, rel = ece_score(y_true, probs)
brier = np.mean(np.sum((probs - yb) ** 2, 1))
ax[1].plot([0, 1], [0, 1], 'k--', lw=.8)
ax[1].plot([r[0] for r in rel], [r[1] for r in rel], 'o-')
ax[1].set_xlabel('Rata-rata keyakinan (probabilitas maks.)'); ax[1].set_ylabel('Akurasi'); ax[1].grid(alpha=.3)
ax[1].set_title(f'Kalibrasi: ECE = {ECE:.3f}, Brier = {brier:.3f}')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/pr_kalibrasi.png', dpi=300); plt.show()
correct = y_pred == y_true
print('Median keyakinan prediksi benar:', np.median(probs.max(1)[correct]).round(3),
      '| salah:', np.median(probs.max(1)[~correct]).round(3) if (~correct).any() else '-')

### A15. Analisis ambang glaukoma dan PPV pada prevalensi skrining (Reviewer A#8d, A#13)
Aturan: prediksi glaukoma bila *p*<sub>glaukoma</sub> ≥ *t*; selain itu, kelas dengan probabilitas tertinggi di antara tiga kelas lain.

In [ ]:
def rule_glaucoma(probs, t):
    other = probs.copy(); other[:, GLAUCOMA] = -1
    return np.where(probs[:, GLAUCOMA] >= t, GLAUCOMA, other.argmax(1))
rows = []
for t in ['argmax', 0.5, 0.4, 0.3, 0.2, 0.15, 0.1, 0.05]:
    yp = probs.argmax(1) if t == 'argmax' else rule_glaucoma(probs, t)
    is_g, pred_g = y_true == GLAUCOMA, yp == GLAUCOMA
    sens = (pred_g & is_g).sum() / is_g.sum(); spec_all = (~pred_g & ~is_g).sum() / (~is_g).sum()
    is_n = y_true == NORMAL; spec_norm = 1 - (pred_g & is_n).sum() / is_n.sum()
    ppv1, npv1 = ppv_npv(sens, spec_all, GLAUCOMA_PREVALENCE); ppv2, _ = ppv_npv(sens, spec_norm, GLAUCOMA_PREVALENCE)
    rows.append((t, pct(sens), pct(spec_all), pct(spec_norm), pct((yp == y_true).mean()), pct(ppv1, 1), pct(ppv2, 1), pct(npv1, 1)))
thr = pd.DataFrame(rows, columns=['Ambang t', 'Sensitivitas glaukoma (%)', 'Spesifisitas OvR (%)', 'Spesifisitas vs normal (%)',
                                  'Akurasi 4 kelas (%)', 'PPV @3,54% (OvR)', 'PPV @3,54% (vs normal)', 'NPV @3,54%'])
thr.to_csv(f'{OUT_DIR}/analisis_ambang_glaukoma.csv', index=False); display(thr)
# Ambang dipilih pada data VALIDASI (sensitivitas glaukoma >= 95% dengan ambang tertinggi), lalu dilaporkan pada data UJI
cand = [t for t in np.round(np.arange(0.50, 0.009, -0.01), 2)
        if ((rule_glaucoma(probs_val, t) == GLAUCOMA) & (y_val == GLAUCOMA)).sum() / max(1, (y_val == GLAUCOMA).sum()) >= 0.95]
if cand:
    t_pilih = float(cand[0]); yp = rule_glaucoma(probs, t_pilih); is_g = y_true == GLAUCOMA; pred_g = yp == GLAUCOMA
    AMBANG = {'t_dipilih_di_validasi': t_pilih, 'sens_glaukoma_uji': float((pred_g & is_g).sum() / is_g.sum()),
              'spes_OvR_uji': float((~pred_g & ~is_g).sum() / (~is_g).sum()), 'akurasi_4_kelas_uji': float((yp == y_true).mean())}
else:
    AMBANG = {'t_dipilih_di_validasi': None}
json.dump(AMBANG, open(f'{OUT_DIR}/ambang_glaukoma_dipilih.json', 'w'), indent=1); print(AMBANG)


### A16. Contoh citra uji yang salah diklasifikasikan (Reviewer A#15)


In [ ]:
wrong = np.nonzero(y_pred != y_true)[0]
tampil = wrong[np.argsort(-probs[wrong].max(1))][:36]      # paling yakin tetapi salah; daftar lengkap ada di CSV
n = len(tampil); cols = 6; rows_ = max(1, math.ceil(n / cols))
plt.figure(figsize=(3 * cols, 3.3 * rows_))
for k, i in enumerate(tampil):
    plt.subplot(rows_, cols, k + 1); plt.imshow(Image.open(eval_files[i]).convert('RGB').resize(IMG_SIZE))
    plt.title(f'Asli: {nama[class_names[y_true[i]]]}\nPrediksi: {nama[class_names[y_pred[i]]]} ({probs[i].max():.2f})', fontsize=8)
    plt.axis('off')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/contoh_salah_klasifikasi.png', dpi=200); plt.show()
pred_df.iloc[wrong].to_csv(f'{OUT_DIR}/daftar_salah_klasifikasi.csv', index=False)
_u = dfB.set_index('file').loc[eval_files].reset_index(); _u['benar'] = (y_pred == y_true)
per_sumber = _u.groupby(['kelas', 'lebar', 'tinggi']).agg(n=('file', 'size'), proporsi_benar=('benar', 'mean')).reset_index()
per_sumber.to_csv(f'{OUT_DIR}/akurasi_per_kelas_dan_ukuran_citra.csv', index=False); print(per_sumber[per_sumber.n >= 8].round(3))


### A17. Grad-CAM (Reviewer A#9)
Gradien dihitung terhadap peta fitur keluaran *backbone* (7 × 7 × 1.280), lalu diteruskan ke blok SE dan kepala klasifikasi memakai lapisan yang sama dengan model terlatih.

In [ ]:
def head_forward(f, m=model):
    names = [l.name for l in m.layers]
    if 'se_scale' in names:
        s = m.get_layer('se_squeeze')(f); s = m.get_layer('se_reshape')(s)
        s = m.get_layer('se_fc2')(m.get_layer('se_fc1')(s)); f = m.get_layer('se_scale')([f, s])
    x = m.get_layer('head_gap')(f)
    x = m.get_layer('head_bn1')(x, training=False); x = m.get_layer('head_fc')(x)
    x = m.get_layer('head_bn2')(x, training=False); x = m.get_layer('head_dropout')(x, training=False)
    return m.get_layer('keluaran')(x)

def gradcam(img, cls=None):
    x = tf.convert_to_tensor(img[None], tf.float32)
    with tf.GradientTape() as tape:
        f = base_model(x, training=False); tape.watch(f)
        p = head_forward(f); cls = int(tf.argmax(p[0])) if cls is None else cls; score = p[:, cls]
    g = tape.gradient(score, f)[0]; w = tf.reduce_mean(g, axis=(0, 1))
    cam = tf.nn.relu(tf.reduce_sum(f[0] * w, -1)).numpy()
    cam = cam / (cam.max() + 1e-8)
    return np.array(Image.fromarray(np.uint8(255 * cam)).resize(IMG_SIZE, Image.BILINEAR)) / 255.0, cls, float(p[0, cls])

# cek konsistensi: jalur Grad-CAM harus menghasilkan probabilitas yang sama dengan model.predict
img0 = load_img(eval_files[0]).numpy()
assert np.allclose(head_forward(base_model(img0[None], training=False)).numpy(), model.predict(img0[None], verbose=0), atol=1e-4)

rng = np.random.default_rng(SEED)
_ok = [np.nonzero((y_true == c) & (y_pred == c))[0] for c in range(NUM_CLASSES)]
pick = [rng.choice(v, min(3, len(v)), replace=False) for v in _ok if len(v)]
pick = [int(v) for v in np.concatenate(pick)] + [int(v) for v in tampil[:6]]
plt.figure(figsize=(18, 3.2 * math.ceil(len(pick) / 6)))
for k, i in enumerate(pick):
    img = load_img(eval_files[i]).numpy(); cam, c, pc = gradcam(img)
    plt.subplot(math.ceil(len(pick) / 6), 6, k + 1); plt.imshow(img.astype('uint8')); plt.imshow(cam, cmap='jet', alpha=.4)
    plt.title(f'Asli: {nama[class_names[y_true[i]]]}\nPred: {nama[class_names[c]]} ({pc:.2f})', fontsize=8); plt.axis('off')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/gradcam.png', dpi=200); plt.show()

### A18. Ukuran model dan waktu inferensi (Reviewer A#14)

In [ ]:
size_mb = os.path.getsize(best_path) / 2**20
def time_inference(device, n=30, batch=1):
    @tf.function
    def f(x):
        with tf.device(device):
            return model(x, training=False)
    x = tf.random.uniform((batch,) + IMG_SIZE + (3,), 0, 255)
    for _ in range(5): f(x).numpy()                     # pemanasan + kompilasi graf
    ts = []
    for _ in range(n):
        t0 = time.perf_counter(); f(x).numpy(); ts.append(time.perf_counter() - t0)
    return 1000 * float(np.median(ts)) / batch          # ms per citra
INFER = {'ukuran_bobot_MB': round(size_mb, 1), 'cpu_ms_per_citra_batch1': round(time_inference('/CPU:0'), 1),
         'cpu_ms_per_citra_batch32': round(time_inference('/CPU:0', n=5, batch=32), 1)}
if gpus:
    INFER['gpu_ms_per_citra_batch1'] = round(time_inference('/GPU:0'), 1); INFER['gpu_ms_per_citra_batch32'] = round(time_inference('/GPU:0', n=10, batch=32), 2)
INFER['cpu'] = platform.processor() or platform.machine(); INFER['jumlah_inti_cpu'] = os.cpu_count()
print(INFER)


### A19. Ringkasan angka untuk mengisi artikel

In [ ]:
res = {'lingkungan': ENV, 'konfigurasi': {'atensi': ATENSI, 'rasio': SE_RATIO, 'seed': SEED, 'pantau': PANTAU, 'patience': PATIENCE, 'epoch_maks': EPOCHS},
       'data': {'latih': len(train_files), 'validasi': len(val_files), 'uji': len(test_files), 'citra_dibuang_label_konflik': int(split_df.grup.isin(konflik).sum())},
       'pelatihan': TRAIN_INFO, 'inferensi': INFER, 'ambang_glaukoma': AMBANG,
       'akurasi': S['akurasi'], 'akurasi_CI': acc_ci, 'macro_f1': S['macro']['f1'], 'macro_f1_CI': BS['macro_f1'],
       'kappa': S['kappa'], 'kappa_CI': BS['kappa'], 'AUC': {('micro' if k == 'micro' else class_names[k]): v for k, v in AUC.items()},
       'sensitivitas': {c: d['sensitivitas'] for c, d in zip(class_names, S['per_class'])},
       'ECE': ECE, 'Brier': brier, 'param_total': n_total, 'param_latih': n_train, 'param_SE': n_se,
       'duplikat_total': int(len(dup)), 'duplikat_lintas_latih_validasi_90_10': int(len(lintas))}
json.dump(res, open(f'{OUT_DIR}/ringkasan_hasil.json', 'w'), indent=1, default=float)
g = S['per_class'][GLAUCOMA]
ribu = lambda v: f'{v:,}'.replace(',', '.')
auc_str = ', '.join(nama[class_names[i]] + ' ' + fmt(AUC[i], 3) for i in range(NUM_CLASSES))
sens_str = ', '.join(nama[c] + ' ' + pct(d['sensitivitas']) for c, d in zip(class_names, S['per_class']))
gpu_str = ('; GPU ' + fmt(INFER['gpu_ms_per_citra_batch1'], 1) + ' ms/citra (batch 1)') if 'gpu_ms_per_citra_batch1' in INFER else ''
print(f"""
=== ANGKA UNTUK ARTIKEL (data UJI, n = {len(y_true)}) ===
Model       : EfficientNetB0 + {ATENSI}{' r=' + str(SE_RATIO) if ATENSI in ('se', 'cbam') else ''}, seed {SEED}
Lingkungan  : TensorFlow {ENV['tensorflow']}, Keras {ENV['keras']}, GPU {ENV['gpu']}
Data        : latih {len(train_files)}, validasi {len(val_files)}, uji {len(test_files)}
Pelatihan   : berhenti pada epoch ke-{TRAIN_INFO['epoch_ditempuh']} ({TRAIN_INFO['berhenti_karena']}), bobot terbaik epoch ke-{TRAIN_INFO['epoch_terbaik']}, waktu {fmt(TRAIN_INFO['waktu_latih_menit'], 1)} menit
Akurasi     : {pct(S['akurasi'])}% (95% CI Wilson {pct(acc_ci[0])} sampai {pct(acc_ci[1])}%), {np.trace(S['cm'])}/{S['cm'].sum()}
Macro F1    : {pct(S['macro']['f1'])}% (95% CI {pct(BS['macro_f1'][0])} sampai {pct(BS['macro_f1'][1])}%)
Kappa       : {fmt(S['kappa'], 3)} (95% CI {fmt(BS['kappa'][0], 3)} sampai {fmt(BS['kappa'][1], 3)})
Sensitivitas: {sens_str}
Spes. glauk.: {pct(g['spesifisitas'])}%
AUC         : {auc_str}; micro {fmt(AUC['micro'], 3)}
Kalibrasi   : ECE {fmt(ECE, 3)}, Brier {fmt(brier, 3)}
Parameter   : total {ribu(n_total)}, dapat dilatih {ribu(n_train)}, blok SE {ribu(n_se)}
Bobot model : {fmt(INFER['ukuran_bobot_MB'], 1)} MB; inferensi CPU {fmt(INFER['cpu_ms_per_citra_batch1'], 1)} ms/citra (batch 1){gpu_str}
Duplikat    : {len(dup)} pasangan pHash (Hamming <= {THR}); {len(lintas)} lintas latih-validasi pada pembagian 90:10 naskah awal
Berkas      : {OUT_DIR}
""")
